# RAG Data Ingestion Pipeline

This notebook reads every document in `DataSets_RAG/`, extracts text, chunks it by legal section boundaries, attaches metadata, embeds with BGE-M3, and stores everything in ChromaDB.

**dataset has:** 72 PDFs, 25 TXTs, 15 HTMLs, 18 misnamed PDFs (`.2008` extension), 16 JSON metadata files  
**Known issues handled:** empty placeholder files, misnamed PDFs, website homepage HTMLs, duplicates

In [1]:
!pip install -q chromadb sentence-transformers pdfplumber pytesseract Pillow beautifulsoup4 lxml tqdm pandas rich

In [2]:
!tesseract --version

tesseract 5.5.3
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.58 : libtiff 4.7.2 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.9 zlib/1.2.12 liblzma/5.8.3 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.4 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.68.1


In [3]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/ritesh/Downloads/IP_Shakti_Sahayak")
DATASET_ROOT = PROJECT_ROOT / "DataSets_RAG"
CHROMA_DB_PATH = PROJECT_ROOT / "chromadb_store"

#  EMBEDDING MODEL

# Option A (Heavy)
# EMBEDDING_MODEL = "BAAI/bge-m3"

# Option B (Lighter & Faster for Macbooks):
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

# ═══════════════════════════════════════════════
#  CHUNKING CONFIG
# ═══════════════════════════════════════════════
MAX_CHUNK_SIZE = 1000       # characters per chunk
CHUNK_OVERLAP  = 200        # overlap between consecutive chunks
MIN_CHUNK_SIZE = 50         # skip chunks shorter than this

# ═══════════════════════════════════════════════
#  CHROMADB
# ═══════════════════════════════════════════════
COLLECTION_NAME = "ip_sakti_legal_corpus"

print(f"📂 Dataset  : {DATASET_ROOT}")
print(f"💾 ChromaDB : {CHROMA_DB_PATH}")
print(f"🧠 Embedding: {EMBEDDING_MODEL}")
print(f"✂️  Chunks   : max={MAX_CHUNK_SIZE}, overlap={CHUNK_OVERLAP}, min={MIN_CHUNK_SIZE}")

📂 Dataset  : /Users/ritesh/Downloads/IP_Shakti_Sahayak/DataSets_RAG
💾 ChromaDB : /Users/ritesh/Downloads/IP_Shakti_Sahayak/chromadb_store
🧠 Embedding: BAAI/bge-small-en-v1.5
✂️  Chunks   : max=1000, overlap=200, min=50


## Metadata Map

Every folder in your `DataSets_RAG/` maps to metadata fields.  
When we encounter a file, we walk **up** its folder path and merge all matching metadata.  
This lets you later filter retrieval like: *"Give me only India + Patent chunks"*.

In [4]:
# Keys = exact folder names from YOUR DataSets_RAG structure
# Values = metadata fields to attach to every chunk from that folder

FOLDER_METADATA = {
    # ── Top-level folders ──
    "INDIAN_LAWS": {
        "jurisdiction": "india",
        "domain": "legislation",
        "authority": "Parliament of India",
        "doc_type": "statute",
    },
    "IP_guidelines": {
        "jurisdiction": "india",
        "domain": "patent",
        "authority": "Indian Patent Office",
        "doc_type": "guideline",
    },
    "ABS_BIODIVERSITY": {
        "jurisdiction": "india",
        "domain": "biodiversity",
        "authority": "National Biodiversity Authority",
        "doc_type": "regulation",
    },
    "AYURVEDA_REGULATION": {
        "jurisdiction": "india",
        "domain": "drug_regulation",
        "authority": "Ministry of AYUSH / CDSCO",
        "doc_type": "regulation",
    },
    "FOOD_REGULATION": {
        "jurisdiction": "india",
        "domain": "food_regulation",
        "authority": "FSSAI",
        "doc_type": "regulation",
    },
    "TRADITIONAL_KNOWLEDGE": {
        "jurisdiction": "india",
        "domain": "traditional_knowledge",
        "authority": "TKDL / CSIR",
        "doc_type": "case_study",
    },
    "INTERNATIONAL": {
        "jurisdiction": "international",
        "domain": "international_ip",
        "authority": "WIPO / WTO / UN",
        "doc_type": "treaty",
    },
    "LEGAL_CORPUS": {
        "jurisdiction": "international",
        "domain": "international_ip",
        "authority": "Various",
        "doc_type": "treaty",
    },

    # ── Sub-folders: override specific fields ──
    "patents_guidelines": {"sub_domain": "patent_examination"},
    "patents_manuals":    {"sub_domain": "patent_procedure"},
    "ABS Regulations":    {"sub_domain": "abs_regulations"},
    "Biodiversity Rules": {"sub_domain": "biodiversity_rules"},
    "NBA Guidelines":     {"sub_domain": "nba_guidelines"},
    "Drugs and Cosmetics Rules": {"sub_domain": "drugs_cosmetics_rules"},
    "First Schedule":     {"sub_domain": "authoritative_texts"},
    "Schedule T":         {"sub_domain": "gmp_ayurveda"},
    "Ministry of AYUSH Guidelines": {"sub_domain": "ayush_guidelines"},
    "Ayurveda Aahar Regulations":   {"sub_domain": "ayurveda_aahar"},
    "TRIPS":              {"authority": "WTO",  "sub_domain": "trips"},
    "Nagoya Protocol":    {"authority": "UN CBD", "sub_domain": "nagoya_protocol"},
    "GRATK Treaty 2024":  {"authority": "WIPO",  "sub_domain": "gratk"},
    "Neem Case":          {"sub_domain": "biopiracy_case"},
    "Turmeric Case":      {"sub_domain": "biopiracy_case"},
    "Basmati Case":       {"sub_domain": "biopiracy_case"},
    "TKDL Public Case Studies": {"sub_domain": "tkdl"},
    "API Public Material":      {"sub_domain": "pharmacopoeia"},

    # ── Filename-level overrides for INDIAN_LAWS (auto-detect domain) ──
}

# Map specific filenames to a more precise domain
FILENAME_DOMAIN_MAP = {
    "patent_act":              "patent",
    "biological_diversity":    "biodiversity",
    "TheDrugsandCosmetics":    "drug_regulation",
    "TheTradeMarks":           "trademark",
    "TheGeographicalIndications": "geographical_indication",
    "Copyright_Act":           "copyright",
    "Designs_Act":             "design",
    "Plant Varieties":         "plant_variety",
    "PPVFR":                   "plant_variety",
    "Protection of Plant":     "plant_variety",
}

print(f"✅ Metadata map defined for {len(FOLDER_METADATA)} folders + {len(FILENAME_DOMAIN_MAP)} filename patterns")

✅ Metadata map defined for 26 folders + 10 filename patterns


## Text Extraction Functions

Handles all file types in your dataset:
- **PDF** (digital text) → `pdfplumber`
- **PDF** (scanned images) → `pdfplumber` + `pytesseract` OCR fallback
- **HTML** (saved web pages) → `BeautifulSoup` (strips CSS/JS/navigation)
- **TXT** → plain read
- **`.2008` / `.2011` files** → treated as PDFs (your `patents_manuals/` folder has these)

In [5]:
import pdfplumber
from bs4 import BeautifulSoup
import json
import re
from typing import Optional, List, Dict


def extract_pdf(file_path: str) -> str:
    """Extract text from a PDF. Falls back to OCR for scanned pages."""
    pages = []
    try:
        with pdfplumber.open(file_path) as pdf:
            for i, page in enumerate(pdf.pages):
                text = page.extract_text()
                if text and len(text.strip()) > 30:
                    pages.append(text)
                else:
                    # Scanned page — try OCR
                    ocr = _try_ocr(page)
                    if ocr:
                        pages.append(ocr)
    except Exception as e:
        print(f"    ⚠️  PDF error: {e}")
    return "\n\n".join(pages)


def _try_ocr(page) -> Optional[str]:
    """OCR a single page using Tesseract. Returns None if unavailable."""
    try:
        import pytesseract
        img = page.to_image(resolution=300).original
        text = pytesseract.image_to_string(img, lang="eng")
        return text.strip() if text and len(text.strip()) > 30 else None
    except Exception:
        return None


def extract_html(file_path: str) -> str:
    """Extract meaningful text from HTML, stripping boilerplate."""
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            soup = BeautifulSoup(f.read(), "lxml")
        # Remove non-content elements
        for tag in soup(["script", "style", "nav", "header", "footer",
                         "noscript", "meta", "link", "iframe"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        return text
    except Exception as e:
        print(f"    ⚠️  HTML error: {e}")
        return ""


def extract_txt(file_path: str) -> str:
    """Read a plain text file."""
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception as e:
        print(f"    ⚠️  TXT error: {e}")
        return ""


def extract_text(file_path: str) -> str:
    """
    Master router — picks the right extractor based on extension.
    Handles the .2008/.2011 misnamed PDFs in your patents_manuals folder.
    """
    ext = Path(file_path).suffix.lower()

    # Known PDF extensions + your misnamed PDFs (.2008, .2011)
    if ext == ".pdf" or ext in (".2008", ".2011"):
        return extract_pdf(file_path)
    elif ext in (".html", ".htm"):
        return extract_html(file_path)
    elif ext in (".txt", ".text", ".md"):
        return extract_txt(file_path)
    else:
        return ""


print("✅ Text extraction functions ready")
print("   Supports: .pdf, .html, .txt, .2008, .2011 (misnamed PDFs)")

✅ Text extraction functions ready
   Supports: .pdf, .html, .txt, .2008, .2011 (misnamed PDFs)


## Text Cleaning

Removes noise from government PDFs (page numbers, repeated headers, excessive whitespace) while preserving legal structure.

In [6]:
def clean_text(text: str) -> str:
    """Clean extracted text while preserving legal section structure."""
    # Remove common PDF noise
    text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'---\s*Page\s+\d+.*?---', '', text)  # Your .txt files have these markers
    text = re.sub(r'(?:THE GAZETTE OF INDIA.*?\n)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'(?:REGISTERED NO\..*?\n)', '', text, flags=re.IGNORECASE)
    
    # Remove Google Patents boilerplate (from your Turmeric/Neem/Basmati .txt files)
    text = re.sub(r'Download PDF\n', '', text)
    text = re.sub(r'Info\nPublication number\n', '', text)
    text = re.sub(r'- Google Patents\n', '', text)
    
    # Collapse excessive whitespace but keep paragraph breaks
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    
    # Strip each line
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    
    return text.strip()


print("✅ Text cleaning function ready")

✅ Text cleaning function ready


## Legal-Aware Chunking

This is the **most critical part** of the entire pipeline.

**Why not just split every 1000 characters?**  
Because legal text has structure. If you blindly split in the middle of Section 3(p) of the Patents Act, you lose the meaning. Instead, we:

1. Split on **legal section boundaries** (Section X, Rule X, Article X, Chapter X)
2. If a section is too long, sub-split at **paragraph boundaries**
3. If still too long, split at **sentence boundaries** with overlap
4. Extract the **section reference** (e.g., "Section 3(p)") as metadata for each chunk

In [7]:
from dataclasses import dataclass, field


@dataclass
class Chunk:
    """A single piece of text with its metadata."""
    content: str
    metadata: Dict[str, str] = field(default_factory=dict)


def extract_section_ref(text: str) -> Optional[str]:
    """Pull out 'Section 3(p)' or 'Rule 12' or 'Article 27' from text."""
    patterns = [
        r'((?:Section|SECTION|Sec\.?)\s+\d+[A-Za-z]*(?:\([a-z0-9]+\))*)',
        r'((?:Rule|RULE)\s+\d+[A-Za-z]*)',
        r'((?:Article|ARTICLE)\s+\d+[A-Za-z]*)',
        r'((?:Chapter|CHAPTER)\s+[IVXLCDM\d]+)',
        r'((?:Schedule|SCHEDULE)\s+[A-Z]+)',
    ]
    for pat in patterns:
        m = re.search(pat, text[:300])  # only scan start of chunk
        if m:
            return m.group(1).strip()
    return ""


def split_with_overlap(text: str, max_size: int, overlap: int) -> List[str]:
    """Split long text at sentence boundaries with overlap."""
    sentences = re.split(r'(?<=[.!?;])\s+', text)
    chunks = []
    current = ""

    for sent in sentences:
        if len(current) + len(sent) + 1 <= max_size:
            current = (current + " " + sent).strip()
        else:
            if current:
                chunks.append(current)
            # Start next chunk with overlap from previous
            if overlap > 0 and current:
                current = current[-overlap:] + " " + sent
            else:
                current = sent

    if current.strip():
        chunks.append(current.strip())
    return chunks


def chunk_document(text: str, base_metadata: Dict[str, str]) -> List[Chunk]:
    """
    Smart chunking that respects legal document structure.
    
    Strategy:
    1. Try to split on Section/Rule/Article/Chapter boundaries
    2. If a section is small enough → one chunk
    3. If too big → sub-split with overlap at sentence boundaries
    """
    chunks = []

    # Pattern to split on legal section headings
    section_re = re.compile(
        r'\n(?='
        r'(?:SECTION|Section|Sec\.?)\s+\d+'
        r'|(?:RULE|Rule)\s+\d+'
        r'|(?:ARTICLE|Article)\s+\d+'
        r'|(?:CHAPTER|Chapter)\s+[IVXLCDM\d]+'
        r'|(?:PART|Part)\s+[IVXLCDM\d]+'
        r'|\d+[\.\)]\s+[A-Z]'
        r')',
        re.MULTILINE
    )

    sections = section_re.split(text)

    # If no legal sections found, fall back to paragraph splitting
    if len(sections) <= 1:
        sections = text.split('\n\n')

    for section in sections:
        section = section.strip()
        if len(section) < MIN_CHUNK_SIZE:
            continue

        if len(section) <= MAX_CHUNK_SIZE:
            # Small enough — keep as one chunk
            meta = {**base_metadata, "section_reference": extract_section_ref(section)}
            chunks.append(Chunk(content=section, metadata=meta))
        else:
            # Too big — sub-split with overlap
            sub_parts = split_with_overlap(section, MAX_CHUNK_SIZE, CHUNK_OVERLAP)
            for i, part in enumerate(sub_parts):
                if len(part) < MIN_CHUNK_SIZE:
                    continue
                meta = {
                    **base_metadata,
                    "section_reference": extract_section_ref(part),
                    "sub_chunk_index": str(i + 1),
                }
                chunks.append(Chunk(content=part, metadata=meta))

    return chunks


print("✅ Legal-aware chunking ready")
print("   Splits on: Section | Rule | Article | Chapter | Part")
print("   Falls back to paragraph → sentence splitting for unstructured text")

✅ Legal-aware chunking ready
   Splits on: Section | Rule | Article | Chapter | Part
   Falls back to paragraph → sentence splitting for unstructured text


## Build Metadata for Each File

Walks up the folder hierarchy to assign `jurisdiction`, `domain`, `authority`, etc.  
Also reads the `metadata.json` files you already have in each subfolder for extra info.

In [8]:
def load_json_metadata(file_path: Path) -> Dict:
    """
    Check if there's a metadata.json in the same folder.
    If so, find the entry matching this filename and return it.
    """
    meta_file = file_path.parent / "metadata.json"
    if not meta_file.exists():
        return {}
    try:
        with open(meta_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            for entry in data:
                if entry.get("filename") == file_path.name:
                    return {
                        "title": entry.get("title", ""),
                        "description": entry.get("description", ""),
                        "original_source": entry.get("source", ""),
                    }
        return {}
    except Exception:
        return {}


def build_metadata(file_path: Path) -> Dict[str, str]:
    """
    Construct metadata for a file by:
    1. Setting defaults
    2. Walking up folder hierarchy and merging from FOLDER_METADATA
    3. Checking filename patterns for more precise domain
    4. Loading companion metadata.json if present
    """
    meta = {
        "source_file": file_path.name,
        "jurisdiction": "india",
        "domain": "general",
        "authority": "Unknown",
        "doc_type": "document",
        "sub_domain": "",
        "section_reference": "",
        "title": "",
        "description": "",
        "original_source": "",
    }

    # Walk up folder path
    try:
        parts = file_path.relative_to(DATASET_ROOT).parts
    except ValueError:
        parts = ()

    for part in parts:
        if part in FOLDER_METADATA:
            meta.update(FOLDER_METADATA[part])

    # Check filename for more precise domain
    fname = file_path.stem
    for pattern, domain in FILENAME_DOMAIN_MAP.items():
        if pattern.lower() in fname.lower():
            meta["domain"] = domain
            break

    # Load companion metadata.json
    json_meta = load_json_metadata(file_path)
    if json_meta:
        meta.update(json_meta)

    return meta


print("✅ Metadata builder ready")
print("   Sources: folder hierarchy + filename patterns + metadata.json")

✅ Metadata builder ready
   Sources: folder hierarchy + filename patterns + metadata.json


## Scan, Extract, Chunk Everything

This walks through your entire `DataSets_RAG/` folder and builds the complete chunk list.  
It automatically **skips**:
- `.DS_Store`, `metadata.json` files
- Files smaller than 100 bytes (the empty placeholders found earlier)
- Files that yield no meaningful text after extraction

In [9]:
from tqdm.notebook import tqdm
import time

# File types to process (including your misnamed PDFs)
SUPPORTED_EXTENSIONS = {".pdf", ".html", ".htm", ".txt", ".text", ".md", ".2008", ".2011"}
SKIP_FILENAMES = {".DS_Store", "metadata.json", ".gitkeep"}
MIN_FILE_SIZE = 100  # bytes — skip empty placeholders


def discover_files() -> List[Path]:
    """Find all processable files in the dataset."""
    files = []
    for root, dirs, filenames in os.walk(DATASET_ROOT):
        for fname in sorted(filenames):
            if fname in SKIP_FILENAMES:
                continue
            fpath = Path(root) / fname
            ext = fpath.suffix.lower()
            if ext not in SUPPORTED_EXTENSIONS:
                continue
            if fpath.stat().st_size < MIN_FILE_SIZE:
                continue
            files.append(fpath)
    return files


# ── Discover all files ──
all_files = discover_files()
print(f"📁 Found {len(all_files)} files to process\n")

# Show breakdown by extension
ext_counts = {}
for f in all_files:
    ext = f.suffix.lower()
    ext_counts[ext] = ext_counts.get(ext, 0) + 1
for ext, count in sorted(ext_counts.items(), key=lambda x: -x[1]):
    print(f"   {ext:8s} → {count} files")


# ── Process each file: extract → clean → chunk ──
all_chunks: List[Chunk] = []
skipped = []
processing_log = []

start_time = time.time()

for fpath in tqdm(all_files, desc="Processing files"):
    rel_path = str(fpath.relative_to(DATASET_ROOT))

    # 1. Build metadata
    metadata = build_metadata(fpath)

    # 2. Extract text
    raw_text = extract_text(str(fpath))

    if not raw_text or len(raw_text.strip()) < MIN_CHUNK_SIZE:
        skipped.append((rel_path, "no text extracted"))
        continue

    # 3. Clean text
    cleaned = clean_text(raw_text)

    if len(cleaned) < MIN_CHUNK_SIZE:
        skipped.append((rel_path, "too short after cleaning"))
        continue

    # 4. Chunk
    chunks = chunk_document(cleaned, metadata)
    all_chunks.extend(chunks)

    processing_log.append({
        "file": rel_path,
        "raw_chars": len(raw_text),
        "clean_chars": len(cleaned),
        "chunks": len(chunks),
        "jurisdiction": metadata.get("jurisdiction", "?"),
        "domain": metadata.get("domain", "?"),
    })

elapsed = time.time() - start_time

print(f"\n{'='*60}")
print(f"📊 PROCESSING COMPLETE — {elapsed:.1f} seconds")
print(f"   Files processed : {len(processing_log)}")
print(f"   Files skipped   : {len(skipped)}")
print(f"   Total chunks    : {len(all_chunks)}")
print(f"{'='*60}")

if skipped:
    print(f"\n⚠️  Skipped files:")
    for path, reason in skipped:
        print(f"   • {path}: {reason}")

📁 Found 125 files to process

   .pdf     → 72 files
   .txt     → 19 files
   .2008    → 18 files
   .html    → 15 files
   .2011    → 1 files


Processing files:   0%|          | 0/125 [00:00<?, ?it/s]

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No


📊 PROCESSING COMPLETE — 1184.3 seconds
   Files processed : 122
   Files skipped   : 3
   Total chunks    : 49689

⚠️  Skipped files:
   • AYURVEDA_REGULATION/Ministry of AYUSH Guidelines/WHO_GMP_Guidelines_Herbal_Medicines.html: no text extracted
   • AYURVEDA_REGULATION/Ministry of AYUSH Guidelines/WHO_Traditional_Medicine_Evaluation_Guidelines.html: no text extracted
   • TRADITIONAL_KNOWLEDGE/API Public Material/Ayurvedic_Pharmacopoeia_of_India_Standards_Overview.html: no text extracted


## Inspect Chunks (Sanity Check)

Before embedding, inspect what the pipeline produced.  
If the numbers look wrong here, fix the problem **before** spending time on embeddings.

In [10]:
import pandas as pd

# ── Processing log table ──
if processing_log:
    df_log = pd.DataFrame(processing_log)
    print("📋 Per-file processing summary:")
    print(df_log.to_string(index=False))
    print()

# ── Chunk distribution ──
if all_chunks:
    chunk_data = []
    for c in all_chunks:
        chunk_data.append({
            "source_file": c.metadata.get("source_file", ""),
            "jurisdiction": c.metadata.get("jurisdiction", ""),
            "domain": c.metadata.get("domain", ""),
            "section_ref": c.metadata.get("section_reference", ""),
            "length": len(c.content),
        })
    df_chunks = pd.DataFrame(chunk_data)

    print("📊 Chunks by Jurisdiction:")
    print(df_chunks["jurisdiction"].value_counts().to_string())
    print()

    print("📊 Chunks by Domain:")
    print(df_chunks["domain"].value_counts().to_string())
    print()

    print(f"📊 Chunk length stats:")
    print(f"   Mean : {df_chunks['length'].mean():.0f} chars")
    print(f"   Min  : {df_chunks['length'].min()} chars")
    print(f"   Max  : {df_chunks['length'].max()} chars")
    print(f"   Total: {df_chunks['length'].sum():,} chars")
    print()

    print("📝 First 5 chunks (preview):")
    for i, c in enumerate(all_chunks[:5]):
        print(f"\n--- Chunk {i+1} ---")
        print(f"  Source : {c.metadata.get('source_file')}")
        print(f"  Domain : {c.metadata.get('domain')}")
        print(f"  Section: {c.metadata.get('section_reference', 'N/A')}")
        print(f"  Content: {c.content[:200]}...")
else:
    print("❌ No chunks! Check your DATASET_ROOT path and file contents.")

📋 Per-file processing summary:
                                                                                                                                                                                                                                                                                                                              file  raw_chars  clean_chars  chunks  jurisdiction                  domain
                                                                                                                                                                                                                                    IP_guidelines/patents_guidelines/Annexure I - Additional Illustrative Examples (CRIs 2025).pdf     128669       127672     188         india                  patent
                                                                                                                                                                                       

## Load Embedding Model


In [11]:
from sentence_transformers import SentenceTransformer
import time

print(f"⏳ Loading embedding model: {EMBEDDING_MODEL}")

start = time.time()
# Added device="mps" to use Mac GPU
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device="mps")
elapsed = time.time() - start

# These are the lines you were missing!
dim = embedding_model.get_sentence_embedding_dimension()
print(f"✅ Model loaded in {elapsed:.1f}s")
print(f"   Embedding dimension: {dim}")

# Quick test
test = embedding_model.encode("Section 3(p) of the Patents Act bars patenting of traditional knowledge")
print(f"   Test vector shape  : {test.shape}")

⏳ Loading embedding model: BAAI/bge-small-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/var/folders/g8/myg8rccn3jvgj765h6v_859c0000gn/T/ipykernel_7627/45212938.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = embedding_model.get_sentence_embedding_dimension()


✅ Model loaded in 19.5s
   Embedding dimension: 384
   Test vector shape  : (384,)


## Initialize ChromaDB

Creates a **persistent** ChromaDB database on disk at `IP_Shakti_Sahayak/chromadb_store/`.  
Data survives kernel restarts. You only need to re-run ingestion if you add new documents.

In [12]:
import chromadb

# Create persistent client
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))

# ⚠️  Uncomment the next 2 lines ONLY if you want to DELETE existing data and start fresh
try: chroma_client.delete_collection(name=COLLECTION_NAME)
except: pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={
        "description": "IP-SAKTI Sahayak — Ayurveda IP & Regulatory Legal Corpus",
        "hnsw:space": "cosine",  # use cosine similarity
    }
)

print(f"✅ ChromaDB initialized")
print(f"   Path      : {CHROMA_DB_PATH}")
print(f"   Collection: {COLLECTION_NAME}")
print(f"   Existing  : {collection.count()} documents")

✅ ChromaDB initialized
   Path      : /Users/ritesh/Downloads/IP_Shakti_Sahayak/chromadb_store
   Collection: ip_sakti_legal_corpus
   Existing  : 0 documents


## Embed & Store All Chunks in ChromaDB

This is the main ingestion cell. It:
1. Embeds chunks in batches of 32
2. Generates a unique ID per chunk (content hash)
3. Stores text + embedding + all metadata in ChromaDB

In [13]:
import hashlib


def make_chunk_id(chunk: Chunk, index: int) -> str:
    """Unique ID = source filename prefix + content hash."""
    content_hash = hashlib.md5(chunk.content.encode()).hexdigest()[:12]
    source = chunk.metadata.get("source_file", "unk")[:25].replace(" ", "_")
    return f"{source}_{content_hash}_{index}"


def ingest_to_chromadb(chunks: List[Chunk], batch_size: int = 32):
    """Embed and store all chunks in ChromaDB."""
    total = len(chunks)
    print(f"🚀 Ingesting {total} chunks into ChromaDB (batch_size={batch_size})")
    print(f"   This may take several minutes...\n")

    ingested = 0

    for i in tqdm(range(0, total, batch_size), desc="Embedding & storing"):
        batch = chunks[i : i + batch_size]

        # Prepare batch
        texts = [c.content for c in batch]
        ids = [make_chunk_id(c, i + j) for j, c in enumerate(batch)]

        # Flatten metadata (ChromaDB needs str/int/float/bool only)
        metadatas = []
        for c in batch:
            m = {}
            for k, v in c.metadata.items():
                if isinstance(v, (str, int, float, bool)):
                    m[k] = v
                else:
                    m[k] = str(v)
            m["content_length"] = len(c.content)
            metadatas.append(m)

        # Embed
        embeddings = embedding_model.encode(
            texts,
            show_progress_bar=False,
            normalize_embeddings=True,
        ).tolist()

        # Store
        collection.upsert(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas,
        )

        ingested += len(batch)

    print(f"\n{'='*60}")
    print(f"✅ INGESTION COMPLETE")
    print(f"   Chunks ingested : {ingested}")
    print(f"   Collection total: {collection.count()}")
    print(f"   Stored at       : {CHROMA_DB_PATH}")
    print(f"{'='*60}")


# ── Run ingestion ──
if all_chunks:
    ingest_to_chromadb(all_chunks, batch_size=32)
else:
    print("❌ No chunks to ingest. Go back and check Cell 8.")

🚀 Ingesting 49689 chunks into ChromaDB (batch_size=32)
   This may take several minutes...



Embedding & storing:   0%|          | 0/1553 [00:00<?, ?it/s]


✅ INGESTION COMPLETE
   Chunks ingested : 49689
   Collection total: 49689
   Stored at       : /Users/ritesh/Downloads/IP_Shakti_Sahayak/chromadb_store


## Corpus Health Check

Verify everything is stored correctly. Shows distribution by jurisdiction, domain, and authority.

In [14]:
def corpus_health_check():
    total = collection.count()
    print(f"{'='*60}")
    print(f"📊 CORPUS HEALTH CHECK")
    print(f"{'='*60}")
    print(f"Total chunks in ChromaDB: {total}")

    if total == 0:
        print("❌ Collection is empty!")
        return

    # Sample to check metadata distribution
    sample = collection.get(limit=min(total, 1000), include=["metadatas"])

    counts = {"jurisdiction": {}, "domain": {}, "authority": {}}
    for meta in sample["metadatas"]:
        for key in counts:
            val = meta.get(key, "unknown")
            counts[key][val] = counts[key].get(val, 0) + 1

    for category, dist in counts.items():
        print(f"\n📌 By {category.title()}:")
        max_val = max(dist.values()) if dist else 1
        for k, v in sorted(dist.items(), key=lambda x: -x[1]):
            bar = "█" * (v * 30 // max_val)
            print(f"   {k:35s} {bar} {v}")

    print(f"\n{'='*60}")
    print(f"✅ Corpus ready for retrieval!")
    print(f"{'='*60}")

corpus_health_check()

📊 CORPUS HEALTH CHECK
Total chunks in ChromaDB: 49689

📌 By Jurisdiction:
   india                               ██████████████████████████████ 1000

📌 By Domain:
   patent                              ██████████████████████████████ 1000

📌 By Authority:
   Indian Patent Office                ██████████████████████████████ 1000

✅ Corpus ready for retrieval!


## Test Retrieval
Supports **filtered search** by jurisdiction and domain.

In [15]:
def search(query: str, n: int = 5,
           jurisdiction: str = None,
           domain: str = None) -> dict:
    """
    Search the legal corpus.
    
    Args:
        query: natural language question
        n: number of results
        jurisdiction: 'india' or 'international' (optional)
        domain: 'patent', 'biodiversity', 'drug_regulation', etc. (optional)
    """
    # Build filter
    where = None
    conditions = []
    if jurisdiction:
        conditions.append({"jurisdiction": {"$eq": jurisdiction}})
    if domain:
        conditions.append({"domain": {"$eq": domain}})
    if len(conditions) == 1:
        where = conditions[0]
    elif len(conditions) > 1:
        where = {"$and": conditions}

    # Embed query
    q_emb = embedding_model.encode(query, normalize_embeddings=True).tolist()

    # Search
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=n,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    return results


def show_results(query: str, results: dict):
    """Pretty-print search results."""
    print(f"\n{'='*70}")
    print(f"🔍 QUERY: {query}")
    print(f"{'='*70}")

    if not results["documents"] or not results["documents"][0]:
        print("❌ No results found.")
        return

    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    )):
        sim = 1 - dist
        badge = "🟢 HIGH" if sim > 0.7 else "🟡 MED" if sim > 0.5 else "🔴 LOW"

        print(f"\n{'─'*50}")
        print(f"📌 Result {i+1}  |  Similarity: {sim:.3f}  |  {badge}")
        print(f"   📄 Source : {meta.get('source_file', '?')}")
        print(f"   🏛️  Authority : {meta.get('authority', '?')}")
        print(f"   🌐 Jurisdiction: {meta.get('jurisdiction', '?')}")
        print(f"   📂 Domain : {meta.get('domain', '?')}")
        sec = meta.get('section_reference', '')
        if sec:
            print(f"   📎 Section: {sec}")
        title = meta.get('title', '')
        if title:
            print(f"   📖 Title  : {title}")
        print(f"   📝 Content:")
        print(f"      {doc[:300]}...")

    print(f"\n{'='*70}\n")


# ═══════════════════════════════════════════════
#  TEST QUERIES
# ═══════════════════════════════════════════════

# Test 1: Patent + traditional knowledge
r = search("Can I patent an Ayurvedic formulation based on traditional knowledge in India?")
show_results("Can I patent an Ayurvedic formulation based on traditional knowledge?", r)

# Test 2: Filtered — India + biodiversity
r = search("What are the ABS obligations for using biological resources?",
           jurisdiction="india", domain="biodiversity")
show_results("ABS obligations for biological resources? [India + Biodiversity]", r)

# Test 3: Filtered — International
r = search("How does the Nagoya Protocol affect herbal product exports?",
           jurisdiction="international")
show_results("Nagoya Protocol and herbal exports? [International]", r)


🔍 QUERY: Can I patent an Ayurvedic formulation based on traditional knowledge?

──────────────────────────────────────────────────
📌 Result 1  |  Similarity: 0.843  |  🟢 HIGH
   📄 Source : Guidelines for Processing of Patent Applications relating to Traditional Knowledge and Biological Material - 2012.pdf
   🏛️  Authority : Indian Patent Office
   🌐 Jurisdiction: india
   📂 Domain : patent
   📝 Content:
      Applications Relating to Traditional
Knowledge and Biological Material

INTELLECTUAL
PROPERTY INDIA

Patents | Designs | Trademarks
Geographical indications

Office of the Controller General of
Patents, Designs & Trademarks

GUIDELINES FOR PROCESSING OF PATENT APPLICATIONS RELATING TO TRADITIONAL
KN...

──────────────────────────────────────────────────
📌 Result 2  |  Similarity: 0.786  |  🟢 HIGH
   📄 Source : Guidelines for Processing of Patent Applications relating to Traditional Knowledge and Biological Material - 2012.pdf
   🏛️  Authority : Indian Patent Office
   🌐 Jurisdict

---

## What Just Built

```
DataSets_RAG/  (PDFs, HTMLs, TXTs, scanned PDFs, misnamed PDFs)
       ↓
Text Extraction  (pdfplumber + OCR + BeautifulSoup)
       ↓
Cleaning  (remove PDF noise, boilerplate, headers)
       ↓
Legal-Aware Chunking  (split on Section/Rule/Article boundaries)
       ↓
Metadata Attachment  (jurisdiction, domain, authority, section ref)
       ↓
BGE-M3 Embeddings  (1024-dim multilingual vectors)
       ↓
ChromaDB  (persistent vector store with metadata filtering)
       ↓
Filtered Semantic Search  (by jurisdiction + domain)
```